In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score
)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/content/kneeoa_data")

print("Using device:", DEVICE)
print("Dataset:", DATA_DIR)

Using device: cuda
Dataset: /content/kneeoa_data


In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/Knee_OA_Project/archive (10).zip")
print("ZIP exists:", ZIP_PATH.exists())
print("ZIP path:", ZIP_PATH)

ZIP exists: True
ZIP path: /content/drive/MyDrive/Knee_OA_Project/archive (10).zip


In [7]:
from pathlib import Path
import zipfile
import shutil

ZIP_PATH = Path("/content/drive/MyDrive/Knee_OA_Project/archive (10).zip")
EXTRACT_DIR = Path("/content/kneeoa_data")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Extraction complete.")
print("Top-level contents:", [p.name for p in EXTRACT_DIR.iterdir()])

Extraction complete.
Top-level contents: ['auto_test', 'train', 'val', 'test']


In [8]:
from pathlib import Path

DATA_DIR = Path("/content/kneeoa_data")
print("DATA_DIR exists:", DATA_DIR.exists())
print("Contents:", [p.name for p in DATA_DIR.iterdir()])

DATA_DIR exists: True
Contents: ['auto_test', 'train', 'val', 'test']


In [9]:
import pandas as pd

CLASS_NAMES = ["0", "1", "2", "3", "4"]
LABEL_MAP = {c: int(c) for c in CLASS_NAMES}

rows = []

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print("Scanning:", split_dir)

    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls

        if not cls_dir.exists():
            print("Missing class folder:", cls_dir)
            continue

        for p in cls_dir.glob("*.png"):
            rows.append({
                "path": str(p),
                "label": LABEL_MAP[cls],
                "split": split
            })

df = pd.DataFrame(rows)

print("Total images found:", len(df))
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

test_df = df[df["split"] == "test"].reset_index(drop=True)
print("Test size:", len(test_df))

Scanning: /content/kneeoa_data/train
Scanning: /content/kneeoa_data/val
Scanning: /content/kneeoa_data/test
Total images found: 8260
label     0     1     2    3    4
split                            
test    639   296   447  223   51
train  2286  1046  1516  757  173
val     328   153   212  106   27
Test size: 1656


In [11]:
from torchvision import transforms

IMG_SIZE = 256
BATCH_SIZE = 32

eval_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [12]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps

class KneeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"])
        img = ImageOps.exif_transpose(img)
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label, row["path"]

In [13]:
test_loader = DataLoader(
    KneeDataset(test_df, eval_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

xb, yb, pb = next(iter(test_loader))
print("Images:", xb.shape)
print("Labels:", yb.shape)
print("Example path:", pb[0])

Images: torch.Size([32, 3, 256, 256])
Labels: torch.Size([32])
Example path: /content/kneeoa_data/test/0/9827858L.png


In [14]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [15]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Knee_OA_Project")

RESNET1_PATH = PROJECT_DIR / "experiments" / "resnet50_fullimg256_wce_v1" / "best_model.pt"
RESNET2_PATH = PROJECT_DIR / "experiments" / "resnet50_fullimg256_wce_v2" / "best_model.pt"
DENSENET_PATH = PROJECT_DIR / "experiments" / "densenet121_fullimg256_wce_v1" / "best_model.pt"

print("RESNET1_PATH exists:", RESNET1_PATH.exists(), RESNET1_PATH)
print("RESNET2_PATH exists:", RESNET2_PATH.exists(), RESNET2_PATH)
print("DENSENET_PATH exists:", DENSENET_PATH.exists(), DENSENET_PATH)

RESNET1_PATH exists: True /content/drive/MyDrive/Knee_OA_Project/experiments/resnet50_fullimg256_wce_v1/best_model.pt
RESNET2_PATH exists: True /content/drive/MyDrive/Knee_OA_Project/experiments/resnet50_fullimg256_wce_v2/best_model.pt
DENSENET_PATH exists: True /content/drive/MyDrive/Knee_OA_Project/experiments/densenet121_fullimg256_wce_v1/best_model.pt


In [16]:
import torch.nn as nn
from torchvision import models

def build_resnet50():
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.fc.in_features, 5)
    )
    return model

def build_densenet121():
    model = models.densenet121(weights=None)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 5)
    )
    return model

In [17]:
model1 = build_resnet50().to(DEVICE)
model2 = build_resnet50().to(DEVICE)
model3 = build_densenet121().to(DEVICE)

ckpt1 = torch.load(RESNET1_PATH, map_location=DEVICE, weights_only=False)
ckpt2 = torch.load(RESNET2_PATH, map_location=DEVICE, weights_only=False)
ckpt3 = torch.load(DENSENET_PATH, map_location=DEVICE, weights_only=False)

model1.load_state_dict(ckpt1["model_state_dict"])
model2.load_state_dict(ckpt2["model_state_dict"])
model3.load_state_dict(ckpt3["model_state_dict"])

model1.eval()
model2.eval()
model3.eval()

print("Loaded model1 epoch:", ckpt1.get("epoch", "N/A"))
print("Loaded model2 epoch:", ckpt2.get("epoch", "N/A"))
print("Loaded model3 epoch:", ckpt3.get("epoch", "N/A"))

Loaded model1 epoch: 7
Loaded model2 epoch: 3
Loaded model3 epoch: 11


In [18]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score
)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

In [19]:
from tqdm.auto import tqdm

@torch.no_grad()
def predict_weighted_ensemble(loader):
    y_true = []
    y_pred = []
    all_paths = []

    for images, labels, paths in tqdm(loader):
        images = images.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits1 = model1(images)
            logits2 = model2(images)
            logits3 = model3(images)

            logits = (
                0.50 * logits1 +
                0.30 * logits2 +
                0.20 * logits3
            )

        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        all_paths.extend(paths)

    return y_true, y_pred, all_paths

In [20]:
y_true_ens, y_pred_ens, paths_ens = predict_weighted_ensemble(test_loader)

ens_metrics = compute_metrics(y_true_ens, y_pred_ens)

print("Weighted 3-model ensemble test metrics:")
for k, v in ens_metrics.items():
    print(f"{k}: {v:.4f}")

  0%|          | 0/52 [00:00<?, ?it/s]

/tmp/ipykernel_305/4016023157.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


Weighted 3-model ensemble test metrics:
accuracy: 0.7011
precision_macro: 0.7009
recall_macro: 0.7117
f1_macro: 0.7050
balanced_accuracy: 0.7117
qwk: 0.8521


In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_true_ens, y_pred_ens, digits=4))

              precision    recall  f1-score   support

           0     0.7628    0.8404    0.7997       639
           1     0.3838    0.3514    0.3668       296
           2     0.7332    0.6577    0.6934       447
           3     0.8036    0.8072    0.8054       223
           4     0.8214    0.9020    0.8598        51

    accuracy                         0.7011      1656
   macro avg     0.7009    0.7117    0.7050      1656
weighted avg     0.6943    0.7011    0.6963      1656



In [22]:
import pandas as pd

ENSEMBLE_DIR = PROJECT_DIR / "experiments" / "weighted_3model_ensemble"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = ENSEMBLE_DIR / "weighted_ensemble_predictions.csv"

pred_df = pd.DataFrame({
    "path": paths_ens,
    "y_true": y_true_ens,
    "y_pred": y_pred_ens
})
pred_df["abs_error"] = (pred_df["y_true"] - pred_df["y_pred"]).abs()

pred_df.to_csv(PRED_PATH, index=False)

print("Saved:", PRED_PATH)
pred_df.head()

Saved: /content/drive/MyDrive/Knee_OA_Project/experiments/weighted_3model_ensemble/weighted_ensemble_predictions.csv


,path,y_true,y_pred,abs_error
0,/content/kneeoa_data/test/0/9827858L.png,0,0,0
1,/content/kneeoa_data/test/0/9030418R.png,0,0,0
2,/content/kneeoa_data/test/0/9594682R.png,0,0,0
3,/content/kneeoa_data/test/0/9733523L.png,0,0,0
4,/content/kneeoa_data/test/0/9789177L.png,0,0,0


In [34]:
SUMMARY_PATH = ENSEMBLE_DIR / "weighted_ensemble_summary.txt"

summary_lines = []
summary_lines.append("Weighted 3-model Ensemble")
summary_lines.append("")
for k, v in ens_metrics.items():
    summary_lines.append(f"{k}: {v:.4f}")

summary_text = "\n".join(summary_lines)

with open(SUMMARY_PATH, "w") as f:
    f.write(summary_text)

print(summary_text)
print("\nSaved:", SUMMARY_PATH)

Weighted 3-model Ensemble

accuracy: 0.7011
precision_macro: 0.7009
recall_macro: 0.7117
f1_macro: 0.7050
balanced_accuracy: 0.7117
qwk: 0.8521

Saved: /content/drive/MyDrive/Knee_OA_Project/experiments/weighted_3model_ensemble/weighted_ensemble_summary.txt
